# Visualisasi Hasil Evaluasi BAB 4.3 — Perbandingan Model LLM

Notebook ini membaca hasil `npm run evaluate` (folder `evaluation/results/run-YYYY-MM-DD-HHMM/`)
dan menampilkannya sebagai chart untuk skripsi (Tabel 4.3–4.9).

**Cara pakai di Google Colab:**
1. Di komputer Anda, **zip** folder hasil run, contoh:
   ```bash
   cd evaluation/results
   zip -r run.zip run-2026-08-21-0003
   ```
2. Jalankan sel di bawah, lalu upload `run.zip` saat diminta.
3. Jalankan semua sel berikutnya (`Runtime > Run all`).

Notebook ini juga bisa dijalankan lokal (bukan Colab) — cukup ubah `RUN_DIR` di sel *"Cari folder run"*
langsung ke path folder hasil run Anda, lalu skip sel upload.

In [ ]:
import json
import sys
import zipfile
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)

## 1. Upload hasil run (khusus Colab)

Jalankan sel ini lalu pilih file `.zip` folder hasil run. Kalau menjalankan notebook ini
lokal (Jupyter biasa), lewati sel ini dan langsung set `RUN_DIR` di sel berikutnya.

In [ ]:
EXTRACT_ROOT = Path("./eval_results")
EXTRACT_ROOT.mkdir(exist_ok=True)

if IN_COLAB:
    from google.colab import files

    uploaded = files.upload()
    for filename in uploaded:
        if filename.lower().endswith(".zip"):
            with zipfile.ZipFile(filename, "r") as zf:
                zf.extractall(EXTRACT_ROOT)
            print(f"Extracted {filename} -> {EXTRACT_ROOT}/")
        else:
            # single file upload (e.g. summary.json alone) also supported
            target = EXTRACT_ROOT / filename
            target.write_bytes(uploaded[filename])
            print(f"Saved {filename} -> {target}")
else:
    print("Bukan Colab — lewati upload, set RUN_DIR manual di sel berikutnya.")

## 2. Cari folder run

Notebook otomatis mencari folder yang berisi `summary.json`. Di Colab, pencarian dimulai dari
`EXTRACT_ROOT` (hasil upload). Kalau dijalankan lokal, pencarian mencoba beberapa lokasi umum
(folder saat ini, `evaluation/results/`, dsb). Kalau tetap tidak ketemu, isi `RUN_DIR_OVERRIDE`
manual di sel ini dengan path folder run Anda.

In [ ]:
RUN_DIR_OVERRIDE = None  # contoh: Path("evaluation/results/run-2026-08-21-0003")


def find_run_dir(search_roots) -> Path:
    for root in search_roots:
        root = Path(root)
        if not root.exists():
            continue
        matches = sorted(root.rglob("summary.json"))
        if matches:
            # newest run first (folder names are sortable: run-YYYY-MM-DD-HHMM)
            return sorted((m.parent for m in matches), reverse=True)[0]
    raise FileNotFoundError(
        "Tidak menemukan summary.json di lokasi mana pun yang dicoba: "
        f"{[str(r) for r in search_roots]}. Set RUN_DIR_OVERRIDE manual di sel ini, "
        "atau pastikan zip yang diupload berisi folder run hasil `npm run evaluate`."
    )


if RUN_DIR_OVERRIDE is not None:
    RUN_DIR = Path(RUN_DIR_OVERRIDE)
elif IN_COLAB:
    RUN_DIR = find_run_dir([EXTRACT_ROOT])
else:
    cwd = Path.cwd()
    candidates = [
        cwd,
        cwd / "evaluation" / "results",
        cwd.parent / "evaluation" / "results",
        cwd.parent.parent / "evaluation" / "results",
    ]
    RUN_DIR = find_run_dir(candidates)

print("RUN_DIR:", RUN_DIR)
msg = f"summary.json tidak ditemukan di {RUN_DIR}"
assert (RUN_DIR / "summary.json").exists(), msg

## 3. Load data

`summary.json` berisi `modelAggregates` (agregat per model) dan `complexityAggregates`
(agregat per model x kategori kompleksitas). `raw-results.jsonl` berisi hasil per kasus uji.

In [ ]:
with open(RUN_DIR / "summary.json", encoding="utf-8") as f:
    summary = json.load(f)

model_df = pd.DataFrame(summary["modelAggregates"])
complexity_df = pd.DataFrame(summary.get("complexityAggregates", []))

raw_records = []
with open(RUN_DIR / "raw-results.jsonl", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            raw_records.append(json.loads(line))
raw_df = pd.DataFrame(raw_records)

print(f"{len(model_df)} model(s), {len(raw_df)} test record(s)")
model_df[["model", "totalTests", "toolAccuracyPct", "parameterAccuracyPct", "avgLatencyMs", "totalCostUsd"]]

## 4. Tabel ringkas per model (Tabel 4.3 style)

In [ ]:
display_cols = [
    "model", "totalTests", "toolCorrectCount", "toolAccuracyPct",
    "parametersChecked", "parametersCorrect", "parameterAccuracyPct",
    "structureValidPct", "avgLatencyMs", "avgTokensPerRequest",
    "totalCostUsd", "totalErrors",
]
model_df[display_cols].round(2)

## 5. Chart — Ketepatan Tool per Model (A_tool)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(model_df["model"], model_df["toolAccuracyPct"], color="#4C72B0")
ax.set_ylabel("Ketepatan Tool (%)")
ax.set_title("Ketepatan Pemilihan Tool per Model (A_tool)")
ax.set_ylim(0, 105)
for bar, val in zip(bars, model_df["toolAccuracyPct"]):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 1.5, f"{val:.1f}%", ha="center")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("chart_tool_accuracy.png")
plt.show()

## 6. Chart — Ketepatan Parameter per Model (A_parameter)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(model_df["model"], model_df["parameterAccuracyPct"], color="#DD8452")
ax.set_ylabel("Ketepatan Parameter (%)")
ax.set_title("Ketepatan Parameter per Model (A_parameter)")
ax.set_ylim(0, 105)
for bar, val in zip(bars, model_df["parameterAccuracyPct"]):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 1.5, f"{val:.1f}%", ha="center")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("chart_parameter_accuracy.png")
plt.show()

## 7. Chart — Perbandingan Multi-Metrik (Tool vs Parameter vs Struktur)

In [ ]:
metrics = ["toolAccuracyPct", "parameterAccuracyPct", "structureValidPct"]
labels = ["Tool (A_tool)", "Parameter (A_parameter)", "Struktur JSON"]

x = range(len(model_df))
width = 0.25
fig, ax = plt.subplots(figsize=(8, 5))
for i, (metric, label) in enumerate(zip(metrics, labels)):
    offset = (i - 1) * width
    ax.bar([xi + offset for xi in x], model_df[metric], width, label=label)

ax.set_xticks(list(x))
ax.set_xticklabels(model_df["model"], rotation=15)
ax.set_ylabel("Persentase (%)")
ax.set_title("Perbandingan Multi-Metrik Ketepatan per Model")
ax.set_ylim(0, 110)
ax.legend()
plt.tight_layout()
plt.savefig("chart_multi_metric.png")
plt.show()

## 8. Chart — Latensi Rata-rata per Model (Tabel 4.5)

Bar menunjukkan rata-rata (avgLatencyMs), garis error menunjukkan rentang min–max.

In [ ]:
avg = model_df["avgLatencyMs"]
lower_err = avg - model_df["minLatencyMs"]
upper_err = model_df["maxLatencyMs"] - avg

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(
    model_df["model"], avg,
    yerr=[lower_err, upper_err], capsize=6,
    color="#55A868",
)
ax.set_ylabel("Latensi (ms)")
ax.set_title("Latensi Rata-rata per Model (min-avg-max)")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("chart_latency.png")
plt.show()

## 9. Chart — Biaya Rata-rata per Request (Tabel 4.7)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(model_df["model"], model_df["avgCostPerRequestUsd"], color="#C44E52")
ax.set_ylabel("Biaya rata-rata per request (USD)")
ax.set_title("Biaya Rata-rata per Request per Model")
for bar, val in zip(bars, model_df["avgCostPerRequestUsd"]):
    ax.text(bar.get_x() + bar.get_width() / 2, val, f"${val:.4f}", ha="center", va="bottom")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("chart_cost.png")
plt.show()

## 10. Chart — Distribusi Jenis Kesalahan per Model (Tabel 4.9)

Stacked bar chart dari `errorCounts` (WRONG_TOOL, INVALID_OR_MISSING_PARAMETER, INVALID_STRUCTURE,
FAILED_CLARIFICATION, UNNECESSARY_TOOL_CALL, OTHER).

In [ ]:
error_types = [
    "WRONG_TOOL", "INVALID_OR_MISSING_PARAMETER", "INVALID_STRUCTURE",
    "FAILED_CLARIFICATION", "UNNECESSARY_TOOL_CALL", "OTHER",
]
error_df = pd.DataFrame(
    {et: model_df["errorCounts"].apply(lambda d: d.get(et, 0)) for et in error_types},
    index=model_df["model"],
)

fig, ax = plt.subplots(figsize=(8, 5))
bottom = pd.Series([0] * len(error_df), index=error_df.index)
colors = plt.cm.Set2.colors
for i, et in enumerate(error_types):
    ax.bar(error_df.index, error_df[et], bottom=bottom, label=et, color=colors[i % len(colors)])
    bottom = bottom + error_df[et]

ax.set_ylabel("Jumlah Kasus")
ax.set_title("Distribusi Jenis Kesalahan per Model")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("chart_error_distribution.png")
plt.show()

error_df

## 11. Chart — Ketepatan Tool per Kategori Kompleksitas (Tabel 4.8)

Grouped bar: sumbu-x kategori (simple/medium/complex/...), warna = model.

In [ ]:
if not complexity_df.empty:
    pivot = complexity_df.pivot(index="category", columns="model", values="toolAccuracyPct")
    category_order = [c for c in ["simple", "medium", "complex", "ambiguous", "invalid"] if c in pivot.index]
    pivot = pivot.reindex(category_order)

    ax = pivot.plot(kind="bar", figsize=(8, 5))
    ax.set_ylabel("Ketepatan Tool (%)")
    ax.set_title("Ketepatan Tool per Kategori Kompleksitas per Model")
    ax.set_ylim(0, 110)
    plt.xticks(rotation=0)
    plt.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig("chart_accuracy_by_category.png")
    plt.show()
else:
    print("complexityAggregates kosong pada summary.json ini.")

## 12. Chart — Latensi per Kategori Kompleksitas (Tabel 4.6)

In [ ]:
if not complexity_df.empty:
    pivot_lat = complexity_df.pivot(index="category", columns="model", values="avgLatencyMs")
    pivot_lat = pivot_lat.reindex(category_order)

    ax = pivot_lat.plot(kind="bar", figsize=(8, 5))
    ax.set_ylabel("Latensi rata-rata (ms)")
    ax.set_title("Latensi Rata-rata per Kategori Kompleksitas per Model")
    plt.xticks(rotation=0)
    plt.legend(title="Model", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.savefig("chart_latency_by_category.png")
    plt.show()

## 13. Chart — Ringkasan Hasil per Kasus (OK vs MISMATCH vs ERROR)

Dari `raw-results.jsonl`: setiap record punya `actualBehavior`/`errorType`. Kita klasifikasikan
tiap record menjadi `OK` (tidak ada errorTypes dan tidak ada errorType API), `MISMATCH`
(ada errorTypes tapi bukan API error), atau `API_ERROR`.

In [ ]:
def outcome_of(row):
    if row.get("errorType") == "API_ERROR":
        return "API_ERROR"
    error_types = row.get("errorTypes") or []
    if len(error_types) > 0:
        return "MISMATCH"
    return "OK"

raw_df["outcome"] = raw_df.apply(outcome_of, axis=1)
outcome_pivot = raw_df.groupby(["model", "outcome"]).size().unstack(fill_value=0)
outcome_pivot = outcome_pivot.reindex(columns=[c for c in ["OK", "MISMATCH", "API_ERROR"] if c in outcome_pivot.columns])

ax = outcome_pivot.plot(kind="bar", stacked=True, figsize=(8, 5),
                         color={"OK": "#55A868", "MISMATCH": "#DD8452", "API_ERROR": "#C44E52"})
ax.set_ylabel("Jumlah Kasus")
ax.set_title("Ringkasan Hasil per Model (OK / MISMATCH / API_ERROR)")
plt.xticks(rotation=15)
plt.legend(title="Hasil")
plt.tight_layout()
plt.savefig("chart_outcome_summary.png")
plt.show()

outcome_pivot

## 14. (Opsional) Unduh semua chart sebagai ZIP

Kalau di Colab, sel ini membungkus semua PNG yang sudah disimpan menjadi satu `charts.zip`
dan langsung men-download-nya ke komputer Anda.

In [ ]:
png_files = sorted(Path(".").glob("chart_*.png"))
zip_path = "charts.zip"
with zipfile.ZipFile(zip_path, "w") as zf:
    for png in png_files:
        zf.write(png)

print(f"{len(png_files)} chart disimpan ke {zip_path}: {[p.name for p in png_files]}")

if IN_COLAB:
    from google.colab import files as colab_files
    colab_files.download(zip_path)